In [ ]:
!pip install -U codecarbon

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.3/365.3 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 92.3 MB/s eta 0:00:00
  Attempting uninstall: psutil
    Found existing installation: psutil 5.9.5
    Uninstalling psutil-5.9.5:
      Successfully uninstalled psutil-5.9.5


In [ ]:
!pip install -U transformers
!pip install -U codecarbon
!pip install -U torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 47.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.3/365.3 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.0 MB/s eta 0:00:00
  Attempting uninstall: psutil
    Found existing installation: psutil 5.9.5
    Uninstalling psutil-5.9.5:
      Successfully uninstalled psutil-5.9.5


In [ ]:
!pip install -U catboost


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.2 MB/s eta 0:00:00


## Local Inference on GPU
Model page: https://huggingface.co/karths/binary_classification_train_TD

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/karths/binary_classification_train_TD)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-classification", model="karths/binary_classification_train_TD")

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("karths/binary_classification_train_TD")
model = AutoModelForSequenceClassification.from_pretrained("karths/binary_classification_train_TD")

In [ ]:
# ====== BENCHMARK TRANSFORMER EFFICIENCY ======
import os
import json
import time
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from codecarbon import EmissionsTracker
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

# Config
TEST_FILE = "/content/sample_data/2024_10IQR_technical_debt_clean_200.csv"
MODEL_NAME = "karths/binary_classification_train_TD"
RESULTS_DIR = "test_results_transformer"
TEXT_COL = "text"
LABEL_COL = "binary_label"

os.makedirs(RESULTS_DIR, exist_ok=True)

print("="*70)
print("BENCHMARK TRANSFORMER (DistilBERT) - INFERENCE EFFICIENCY")
print("="*70)

# Carica dataset
df_test = pd.read_csv(TEST_FILE)
df_test.columns = [c.strip() for c in df_test.columns]
df_test[TEXT_COL] = df_test[TEXT_COL].astype(str).fillna("").str.strip()
df_test = df_test[df_test[TEXT_COL].str.len() > 0].copy()

print(f"Dataset test: {len(df_test)} samples")

# Carica modello e tokenizer
print("\nCaricamento modello transformer...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

print(f"Device: {device}")
print(f"Modello: {MODEL_NAME}")

# Prepara testi
texts = df_test[TEXT_COL].tolist()
y_true = df_test[LABEL_COL].astype(int).values if LABEL_COL in df_test.columns else None

# ====== INFERENZA CON TRACKING ======
print("\n" + "="*70)
print("INFERENZA CON CARBON TRACKING")
print("="*70)

tracker = EmissionsTracker(
    project_name="transformer_inference",
    output_dir=RESULTS_DIR,
    log_level="error"
)
tracker.start()

start_time = time.time()
predictions = []
probabilities = []

# Inferenza batch-wise per gestire limiti di memoria
batch_size = 32
with torch.no_grad():
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)

        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.nn.functional.softmax(logits, dim=-1)

        batch_preds = torch.argmax(logits, dim=-1).cpu().numpy()
        batch_probs = probs[:, 1].cpu().numpy()  # Probabilità classe TD (1)

        predictions.extend(batch_preds)
        probabilities.extend(batch_probs)

inference_time = time.time() - start_time
emissions_kg = tracker.stop()

predictions = np.array(predictions)
probabilities = np.array(probabilities)

print(f"✓ Inferenza completata!")
print(f"  Tempo totale: {inference_time:.6f} s")
print(f"  Tempo/sample: {(inference_time / len(df_test)) * 1000:.3f} ms")
print(f"  Throughput: {len(df_test) / inference_time:.2f} samples/sec")
print(f"  CO2: {emissions_kg * 1000:.6f} g ({(emissions_kg / len(df_test)) * 1e6:.6f} mg/sample)")

# ====== METRICHE ======
metrics = {
    "model_name": "DistilBERT Teacher",
    "model_source": MODEL_NAME,
    "device": str(device),
    "batch_size": batch_size,
    "test_samples": len(df_test),
    "inference_time_seconds": inference_time,
    "inference_time_ms_per_sample": (inference_time / len(df_test)) * 1000,
    "throughput_samples_per_sec": len(df_test) / inference_time,
    "inference_emissions_kg": float(emissions_kg),
    "inference_emissions_g": float(emissions_kg * 1000),
    "inference_emissions_mg_per_sample": float((emissions_kg / len(df_test)) * 1e6),
}

if y_true is not None:
    # Calcola metriche di classificazione
    accuracy = accuracy_score(y_true, predictions)
    precision = precision_score(y_true, predictions, zero_division=0)
    recall = recall_score(y_true, predictions, zero_division=0)
    f1 = f1_score(y_true, predictions, zero_division=0)
    roc_auc = roc_auc_score(y_true, probabilities)
    pr_auc = average_precision_score(y_true, probabilities)

    metrics.update({
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
    })

    # Efficiency ratios
    efficiency_temporal = f1 / metrics["inference_time_ms_per_sample"]
    efficiency_carbon = f1 / metrics["inference_emissions_mg_per_sample"]

    metrics["efficiency_f1_per_ms"] = float(efficiency_temporal)
    metrics["carbon_efficiency_f1_per_mg_co2"] = float(efficiency_carbon)

    print(f"\n  F1-Score: {f1:.6f}")
    print(f"  ROC-AUC: {roc_auc:.6f}")
    print(f"  Efficiency (F1/ms): {efficiency_temporal:.6f}")
    print(f"  Carbon Efficiency (F1/mg CO2): {efficiency_carbon:.6f}")

# Salva metriche
with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

# Salva predizioni
df_test['pred_label'] = predictions
df_test['pred_prob'] = probabilities
df_test.to_csv(os.path.join(RESULTS_DIR, "predictions.csv"), index=False)

print("\n" + "="*70)
print("BENCHMARK COMPLETATO")
print("="*70)
print(f"Metriche salvate in: {RESULTS_DIR}/metrics.json")
print(f"Predizioni salvate in: {RESULTS_DIR}/predictions.csv")

BENCHMARK TRANSFORMER (DistilBERT) - INFERENCE EFFICIENCY
Dataset test: 200 samples

Caricamento modello transformer...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Device: cuda
Modello: karths/binary_classification_train_TD

INFERENZA CON CARBON TRACKING
✓ Inferenza completata!
  Tempo totale: 2.862136 s
  Tempo/sample: 14.311 ms
  Throughput: 69.88 samples/sec
  CO2: 0.032133 g (0.160666 mg/sample)

  F1-Score: 0.800000
  ROC-AUC: 0.868650
  Efficiency (F1/ms): 0.055902
  Carbon Efficiency (F1/mg CO2): 4.979260

BENCHMARK COMPLETATO
Metriche salvate in: test_results_transformer/metrics.json
Predizioni salvate in: test_results_transformer/predictions.csv


In [ ]:
# ====== BENCHMARK CATBOOST DISTILLED - INFERENCE EFFICIENCY ======
import os
import json
import time
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from codecarbon import EmissionsTracker
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

# Config
TEST_FILE = "/content/sample_data/2024_10IQR_technical_debt_clean_200.csv"
MODEL_FILE = "/content/sample_data/catboost_td_distilled.cbm"
META_FILE = "/content/sample_data/catboost_td_distilled_meta.json"
RESULTS_DIR = "test_results_catboost_distilled"
LABEL_COL = "binary_label"

os.makedirs(RESULTS_DIR, exist_ok=True)

print("="*70)
print("BENCHMARK CATBOOST DISTILLED - INFERENCE EFFICIENCY")
print("="*70)

# Carica dataset
df_test = pd.read_csv(TEST_FILE)
df_test.columns = [c.strip() for c in df_test.columns]

# IMPORTANTE: rinomina "text" -> "orig_text" per compatibilità con il modello
df_test = df_test.rename(columns={"text": "orig_text"})
TEXT_COL = "orig_text"

df_test[TEXT_COL] = df_test[TEXT_COL].astype(str).fillna("").str.strip()
df_test = df_test[df_test[TEXT_COL].str.len() > 0].copy()

print(f"Dataset test: {len(df_test)} samples")

# Carica modello CatBoost
print("\nCaricamento modello CatBoost...")
model = CatBoostRegressor()
model.load_model(MODEL_FILE)
print(f"Modello: {MODEL_FILE}")

# Carica metadata del training
training_meta = {}
if os.path.exists(META_FILE):
    with open(META_FILE, 'r', encoding='utf-8') as f:
        training_meta = json.load(f)
    print(f"\nInfo training:")
    print(f"  Addestrato su colonna: {training_meta.get('text_col')}")
    if 'roc_auc_valid' in training_meta:
        print(f"  ROC-AUC validation: {training_meta.get('roc_auc_valid'):.6f}")
        print(f"  F1-Score validation: {training_meta.get('f1_score'):.6f}")

# Prepara dati per inferenza
y_true = df_test[LABEL_COL].astype(int).values if LABEL_COL in df_test.columns else None

# ====== INFERENZA CON TRACKING ======
print("\n" + "="*70)
print("INFERENZA CON CARBON TRACKING")
print("="*70)

# Crea Pool con text feature (nome deve coincidere con training!)
test_pool = Pool(
    data=df_test[[TEXT_COL]],
    text_features=[TEXT_COL]
)

tracker = EmissionsTracker(
    project_name="catboost_distilled_inference",
    output_dir=RESULTS_DIR,
    log_level="error"
)
tracker.start()

start_time = time.time()
pred_logit = model.predict(test_pool)
inference_time = time.time() - start_time
emissions_kg = tracker.stop()

# Converti logit a probabilità
pred_prob = 1.0 / (1.0 + np.exp(-pred_logit))

print(f"✓ Inferenza completata!")
print(f"  Tempo totale: {inference_time:.6f} s")
print(f"  Tempo/sample: {(inference_time / len(df_test)) * 1000:.3f} ms")
print(f"  Throughput: {len(df_test) / inference_time:.2f} samples/sec")
print(f"  CO2: {emissions_kg * 1000:.6f} g ({(emissions_kg / len(df_test)) * 1e6:.6f} mg/sample)")

# ====== METRICHE ======
metrics = {
    "model_name": "CatBoost Distilled",
    "model_file": MODEL_FILE,
    "test_samples": len(df_test),
    "inference_time_seconds": float(inference_time),
    "inference_time_ms_per_sample": float((inference_time / len(df_test)) * 1000),
    "throughput_samples_per_sec": float(len(df_test) / inference_time),
    "inference_emissions_kg": float(emissions_kg),
    "inference_emissions_g": float(emissions_kg * 1000),
    "inference_emissions_mg_per_sample": float((emissions_kg / len(df_test)) * 1e6),
}

if y_true is not None:
    # Usa soglia dal training o default
    threshold = training_meta.get("best_threshold", 0.5) if training_meta else 0.5
    pred_label = (pred_prob >= threshold).astype(int)

    # Calcola metriche di classificazione
    accuracy = accuracy_score(y_true, pred_label)
    precision = precision_score(y_true, pred_label, zero_division=0)
    recall = recall_score(y_true, pred_label, zero_division=0)
    f1 = f1_score(y_true, pred_label, zero_division=0)
    roc_auc = roc_auc_score(y_true, pred_prob)
    pr_auc = average_precision_score(y_true, pred_prob)
    cm = confusion_matrix(y_true, pred_label)

    metrics.update({
        "threshold": float(threshold),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "confusion_matrix": cm.tolist(),
        "true_negatives": int(cm[0, 0]),
        "false_positives": int(cm[0, 1]),
        "false_negatives": int(cm[1, 0]),
        "true_positives": int(cm[1, 1])
    })

    # Efficiency ratios
    efficiency_temporal = f1 / metrics["inference_time_ms_per_sample"]
    efficiency_carbon = f1 / metrics["inference_emissions_mg_per_sample"]

    metrics["efficiency_f1_per_ms"] = float(efficiency_temporal)
    metrics["carbon_efficiency_f1_per_mg_co2"] = float(efficiency_carbon)

    print(f"\n  Soglia: {threshold:.6f}")
    print(f"  Accuracy: {accuracy:.6f}")
    print(f"  Precision: {precision:.6f}")
    print(f"  Recall: {recall:.6f}")
    print(f"  F1-Score: {f1:.6f}")
    print(f"  ROC-AUC: {roc_auc:.6f}")
    print(f"  PR-AUC: {pr_auc:.6f}")
    print(f"\n  Efficiency (F1/ms): {efficiency_temporal:.6f}")
    print(f"  Carbon Efficiency (F1/mg CO2): {efficiency_carbon:.6f}")

    print(f"\nConfusion Matrix:")
    print(cm)

# Salva metriche
with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

# Salva predizioni
df_test['pred_logit'] = pred_logit
df_test['pred_prob'] = pred_prob
if y_true is not None:
    df_test['pred_label'] = pred_label
df_test.to_csv(os.path.join(RESULTS_DIR, "predictions.csv"), index=False)

print("\n" + "="*70)
print("BENCHMARK COMPLETATO")
print("="*70)
print(f"Metriche salvate in: {RESULTS_DIR}/metrics.json")
print(f"Predizioni salvate in: {RESULTS_DIR}/predictions.csv")

BENCHMARK CATBOOST DISTILLED - INFERENCE EFFICIENCY
Dataset test: 200 samples

Caricamento modello CatBoost...
Modello: /content/sample_data/catboost_td_distilled.cbm

INFERENZA CON CARBON TRACKING
✓ Inferenza completata!
  Tempo totale: 0.038250 s
  Tempo/sample: 0.191 ms
  Throughput: 5228.79 samples/sec
  CO2: 0.000244 g (0.001222 mg/sample)

  Soglia: 0.500000
  Accuracy: 0.815000
  Precision: 0.800000
  Recall: 0.840000
  F1-Score: 0.819512
  ROC-AUC: 0.889000
  PR-AUC: 0.901248

  Efficiency (F1/ms): 4.285061
  Carbon Efficiency (F1/mg CO2): 670.737619

Confusion Matrix:
[[79 21]
 [16 84]]

BENCHMARK COMPLETATO
Metriche salvate in: test_results_catboost_distilled/metrics.json
Predizioni salvate in: test_results_catboost_distilled/predictions.csv
